# 🧠 Augmented Factorized NS-ARC Experiment

**Phase 0:** Pre-Train Factorized Codebooks (Shape $K=256$, Color $K=16$) to create a universal ARC geometry dictionary.
**Transition:** Farthest-Point Sampling (FPS) to extract verified semantic concepts.
**Phase 1:** Train the Slotted-JEPA with RoPE + Sobel + One-Hots, initialized with the semantic concepts.


## 1. Setup & Imports

In [ ]:
import sys, os, torch
import torch.nn as nn
import numpy as np
import wandb
import matplotlib.pyplot as plt
from tqdm import tqdm

# Import Local Project Files
sys.path.insert(0, os.path.abspath('.'))
from arc_data.rearc_dataset import ReARCDataset
from arc_data.arc_dataset import ARCDataset
from analysis.evaluator import run_validation_epoch
from analysis.plot_utils import plot_reconstruction_dashboard, plot_slot_masks

# Import Modular Blocks
from modules.vq import FactorizedVectorQuantizer
from modules.semantic_encoders import SemanticSlotEncoder
from modules.semantic_decoders import SemanticDecoder
from modules.encoders import DeepTransformerEncoder
from modules.decoders import TransformerDecoder
from modules.encoders import CNNEncoder # For Phase 0 Dummy

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

In [ ]:
BASE_CFG = {
    'device': DEVICE,
    'in_channels': 1,
    'input_dim': 64,
    'patch_size': 2,
    'hidden_dim': 128,
    'latent_dim': 128,
    'num_slots': 10,
    'slot_iters': 3,
    'slot_temperature': 0.1,
    'vocab_size': 10,
    'grid_size': 30,
    'focal_gamma': 2.0,
}

train_dataset = ReARCDataset(data_path='arc_data/re-arc')
eval_dataset = ARCDataset(data_path='arc_data/re-arc/arc_original/evaluation')
os.makedirs("evaluation_reports/plots", exist_ok=True)

## 2. Phase 0: Factorized Codebook Pretraining

In [ ]:
class DummyAutoencoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.encoder = CNNEncoder(config)
        self.vq = FactorizedVectorQuantizer(num_shape_codes=256, num_color_codes=16, embedding_dim=128)
        self.decoder = TransformerDecoder(config)
        self.to(DEVICE)
        
    def forward(self, inputs):
        z = self.encoder(inputs)["latent"]
        # VQ Forward
        z_q, vq_loss, p_shape, p_color = self.vq(z)
        out = self.decoder({"latent": z_q})
        return out, vq_loss, p_shape, p_color

def train_phase_0(epochs=100):
    print("\n🚀 Starting Phase 0: Factorized Codebook Pretraining")
    model = DummyAutoencoder(BASE_CFG)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    
    start_epoch = 1
    if os.path.exists("latest_phase0_checkpoint.pth"):
        print("📥 Resuming Phase 0 from checkpoint...")
        checkpoint = torch.load("latest_phase0_checkpoint.pth", map_location=DEVICE, weights_only=False)
        model.load_state_dict(checkpoint['model'])
        opt.load_state_dict(checkpoint['opt'])
        start_epoch = checkpoint['epoch'] + 1

    for epoch in range(start_epoch, epochs+1):
        model.train()
        epoch_losses, vq_losses, p_shapes, p_colors = [], [], [], []
        for _ in tqdm(range(100), desc=f"P0 Epoch {epoch}", leave=False):
            batch = train_dataset.sample(128)
            state = batch['state'].to(DEVICE)
            
            out, vq_loss, p_shape, p_color = model({"state": state})
            loss_dict = model.decoder.loss({"state": state}, out)
            total_loss = loss_dict["loss"] + vq_loss
            
            opt.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            
            epoch_losses.append(loss_dict["loss"].item())
            vq_losses.append(vq_loss.item())
            p_shapes.append(p_shape.item())
            p_colors.append(p_color.item())
            
        avg_recon = np.mean(epoch_losses)
        avg_vq = np.mean(vq_losses)
        avg_p_shape = np.mean(p_shapes)
        avg_p_color = np.mean(p_colors)
        
        wandb.log({
            "P0/Recon_Loss": avg_recon, 
            "P0/VQ_Loss": avg_vq, 
            "P0/Perplexity_Shape_256": avg_p_shape, 
            "P0/Perplexity_Color_16": avg_p_color
        }, step=epoch)
        
        if epoch % 10 == 0:
            torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'epoch': epoch}, "latest_phase0_checkpoint.pth")
            print(f"Epoch {epoch} | Recon: {avg_recon:.4f} | VQ: {avg_vq:.4f} | Shape Util: {avg_p_shape:.1f}/256 | Color Util: {avg_p_color:.1f}/16")
            
            # Marker Checks for Baking
            if avg_p_shape > 240.0 and avg_p_color > 15.0 and avg_recon < 0.05:
                print("✅ Universal Codebook is Fully Baked! Early Stopping Phase 0.")
                break
                
    return model.vq.eval() # Return the frozen Codebook map

## 3. Phase 1: Slotted JEPA Training with Baseline Comparison

In [ ]:
def run_full_pipeline():
    if os.environ.get('WANDB_API_KEY'): wandb.login(key=os.environ.get('WANDB_API_KEY'))
    wb_run = wandb.init(project="NS-ARC-Scaling", name="Factorized-FPS-AugmentedSlots", config=BASE_CFG, reinit=True)
    
    # 1. Pretrain the VQ
    frozen_vq = train_phase_0(epochs=80)
    
    # 2. Extract Farthest Point Samples (FPS)
    print("\n🔍 Extracting 10 FPS Semantic Slots from Frozen Codebooks...")
    semantic_slot_initializers = frozen_vq.get_farthest_point_samples(target_slots=BASE_CFG['num_slots'])
    
    # 3. Initialize Phase 1 Slotted Model
    print("🚀 Initializing Semantic Slot Attention...\n")
    slotted_encoder = SemanticSlotEncoder(BASE_CFG)
    slotted_decoder = SemanticDecoder(BASE_CFG)
    slotted_encoder.inject_semantic_priors(semantic_slot_initializers)
    
    # 4. Initialize Baseline Control Model
    baseline_encoder = DeepTransformerEncoder(BASE_CFG)
    baseline_decoder = TransformerDecoder(BASE_CFG)
    
    opt_slotted = torch.optim.AdamW(list(slotted_encoder.parameters()) + list(slotted_decoder.parameters()), lr=1e-4)
    opt_baseline = torch.optim.AdamW(list(baseline_encoder.parameters()) + list(baseline_decoder.parameters()), lr=1e-4)
    
    EPOCHS = 1000
    STEPS_PER_EPOCH = 200
    BATCH_SIZE = 128
    
    for epoch in range(1, EPOCHS + 1):
        slotted_encoder.train(); slotted_decoder.train()
        baseline_encoder.train(); baseline_decoder.train()
        
        epoch_losses, epoch_recon, epoch_vicreg = [], [], []
        base_losses = []
        
        for _ in tqdm(range(STEPS_PER_EPOCH), desc=f"P1 Epoch {epoch}", leave=False):
            batch = train_dataset.sample(BATCH_SIZE)
            state_batch = batch['state'].to(DEVICE)
            
            # -- Slotted Step --
            z_dict = slotted_encoder({"state": state_batch})
            out_dict = slotted_decoder({"latent": z_dict["latent"]})
            loss_dict = slotted_decoder.loss({"state": state_batch, "latent": z_dict["latent"]}, out_dict)
            opt_slotted.zero_grad()
            loss_dict["loss"].backward()
            opt_slotted.step()
            
            epoch_losses.append(loss_dict["loss"].item())
            epoch_recon.append(loss_dict["recon_loss"].item())
            epoch_vicreg.append(loss_dict["vic_loss"].item())
            
            # -- Baseline Step --
            b_z = baseline_encoder({"state": state_batch})
            b_out = baseline_decoder({"latent": b_z["latent"]})
            b_loss_dict = baseline_decoder.loss({"state": state_batch}, b_out)
            opt_baseline.zero_grad()
            b_loss_dict["loss"].backward()
            opt_baseline.step()
            base_losses.append(b_loss_dict["loss"].item())
            
        # Logging
        wb_run.log({
            "P1_Slot/Total": np.mean(epoch_losses),
            "P1_Slot/Recon": np.mean(epoch_recon),
            "P1_Slot/VICReg": np.mean(epoch_vicreg),
            "P1_Base/Recon": np.mean(base_losses)
        }, step=epoch+100) # Offset for phase 0
        
        # 10-EPOCH VALIDATION INTERVAL
        if epoch % 10 == 0 or epoch == 1:
            print(f"\n[Epoch {epoch:03d}] Slot_Recon: {np.mean(epoch_recon):.4f} | Base_Recon: {np.mean(base_losses):.4f}")
            
            modules_slotted = {'encoder': slotted_encoder, 'decoder': slotted_decoder, 'config': BASE_CFG}
            modules_base = {'encoder': baseline_encoder, 'decoder': baseline_decoder, 'config': BASE_CFG}
            
            s_val_loss, s_val_acc, s_val_perf = run_validation_epoch(modules_slotted, eval_dataset, phase='ae', batch_size=8, device=DEVICE)
            b_val_loss, b_val_acc, b_val_perf = run_validation_epoch(modules_base, eval_dataset, phase='ae', batch_size=8, device=DEVICE)
            
            print(f"--> [Slot Val]  Pixel.Acc: {s_val_acc:.2f}% | Perfect_Regen: {s_val_perf:.2f}%")
            print(f"--> [Base Val]  Pixel.Acc: {b_val_acc:.2f}% | Perfect_Regen: {b_val_perf:.2f}%\n")
            
            wb_run.log({
                "Val_Slot/Pixel_Acc": s_val_acc, "Val_Slot/Perfect": s_val_perf,
                "Val_Base/Pixel_Acc": b_val_acc, "Val_Base/Perfect": b_val_perf
            }, step=epoch+100)
            
            # Visualization & W&B Artifacts
            slotted_encoder.eval(); slotted_decoder.eval()
            with torch.no_grad():
                val_batch = eval_dataset.sample(4)
                val_state = val_batch['state'].to(DEVICE)
                z_val = slotted_encoder({"state": val_state})
                out_val = slotted_decoder({"latent": z_val["latent"]})
                
                z_flat = z_val["latent"].view(4 * BASE_CFG['num_slots'], -1).cpu().numpy()
                recon_grid = out_val["reconstruction"].argmax(dim=1).float()
                
                viz_path = f"evaluation_reports/plots/recon_epoch_{epoch}.png"
                mask_path = f"evaluation_reports/plots/masks_epoch_{epoch}.png"
                plot_reconstruction_dashboard(val_state[0, 0].cpu(), recon_grid[0].cpu(), torch.tensor(z_flat), epoch, viz_path)
                plot_slot_masks(out_val["alphas"], epoch, mask_path)
                
                wb_run.log({
                    "Visuals/Slot_Reconstruction": wandb.Image(viz_path),
                    "Visuals/Slot_Masks": wandb.Image(mask_path)
                }, step=epoch+100)
            
            # Save Checkpoint Artifact
            checkpoint = {'encoder': slotted_encoder.state_dict(), 'decoder': slotted_decoder.state_dict()}
            torch.save(checkpoint, "latest_slotted_checkpoint.pth")
            if epoch % 100 == 0:
                artifact = wandb.Artifact(f'slotted_model_ep{epoch}', type='model')
                artifact.add_file('latest_slotted_checkpoint.pth')
                wb_run.log_artifact(artifact)

    wb_run.finish()
    print("✅ Run Complete.")


In [ ]:
# Execute the architecture
if __name__ == "__main__":
    # Note: Requires WandB environment variables setup in terminal prior to execution.
    run_full_pipeline()
